# Exp 34: ConvNeXt-Tiny

Card: [exp34_convnext_tiny.md](../../experiments/exp34_convnext_tiny.md)


In [ ]:
import os

import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, precision_recall_curve

from core.config import CLASSES, ROOT
from core.cv import VIP, CVResult, Config, CrossValidator
from core.gradcam import GradCamFigures
from core.tracking import Experiment

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    OVERRIDES = {
        "experiment": "cnn-model-convnext-tiny-smoke",
        "folds": [0, 1],
        "epochs": 1,
        "img_size": 64,
        "n_other": 20,
        "num_workers": 0,
        "retrain": True,
        "log_every_n_steps": 1,
    }
    N_POSITIVES = 2  # images per VIP Grad-CAM figure
else:
    OVERRIDES = {}
    N_POSITIVES = None  # all labeled positives
CONFIG_DIR = ROOT / "sections" / "model" / "configs"
CONFIG = Config.from_toml(
    CONFIG_DIR / "exp34_convnext_tiny" / "convnext_tiny.toml", **OVERRIDES
)
EXP26 = Config.from_toml(CONFIG_DIR / "exp26_coarse_dropout_5fold" / "resnet18.toml")
FIGURES = ROOT / "sections" / "model" / "figures"
PREFIX = "exp34_convnext_tiny"
SAVE_FIGURES = not SMOKE

# dataviz reference palette, light mode
SERIES = {"exp26": "#eb6834", "ConvNeXt-Tiny": "#2a78d6"}
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.titlesize": "medium",
        "axes.titlecolor": "#52514e",
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)


## Training


In [ ]:
results = {"exp26": CVResult.load(EXP26), "ConvNeXt-Tiny": CrossValidator(CONFIG).run()}

## Precision-recall curves


In [ ]:
fig, axes = plt.subplots(
    1, len(CLASSES), figsize=(3.4 * len(CLASSES), 3.6), sharey=True
)
for cls, ax in zip(CLASSES, axes):
    for name, r in results.items():
        y, p = r.oof[cls], r.oof[f"p_{cls}"]
        precision, recall, _ = precision_recall_curve(y, p)
        ax.plot(
            recall,
            precision,
            color=SERIES[name],
            lw=2,
            drawstyle="steps-post",
            label=f"{name}, AP {average_precision_score(y, p):.3f}",
        )
    ax.set(
        title=f"{cls} ({int(y.sum())} positives)",
        xlabel="recall",
        xlim=(0, 1.02),
        ylim=(0, 1.05),
    )
    ax.legend(loc="lower left", alignment="left")
axes[0].set_ylabel("precision")
fig.suptitle("Out-of-fold rows of folds 0–4", color="#52514e")
fig.tight_layout()
figures = {f"{PREFIX}_pr_curves.png": fig}
plt.show()

## Grad-CAM VIP: unseen images


In [ ]:
cam = GradCamFigures(CONFIG)
for cls in VIP:
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_validation.png"] = cam.positives(
        cls, n=N_POSITIVES
    )
plt.show()

In [ ]:
with Experiment(CONFIG.experiment).run("summary", {"val_mode": "full"}) as run:
    if SAVE_FIGURES:
        run.log_figures(figures, FIGURES)